# Climate Risk Disclosure Scrapping
This webscrapping tool is developed to download data from the Climate Reporting Entity (CRE) Search Hub. 
* `pdfs_2024` The first analysis was conducted in June 2024 for 35 companies (including 30 companies reported as of 7 May 2024 and 5 extra banks from 2022-2023) 
* `pdfs_2026` The second analysis was conducted in May 2026 for 198 companies (across all target years in 2023-2026)

The CRE search hub is this link: https://crd-app.companiesoffice.govt.nz/dashboard/



In [1]:
#!pip install "selenium>=4.18.1"
#!pip install beautifulsoup4
#!pip install webdriver_manager
#!pip install requests
#!pip install --upgrade selenium

In [1]:
import selenium
selenium.__version__

'4.43.0'

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service as ChromeService
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import ast, re, time, os, requests
import pandas as pd
import numpy as np
from datetime import date

In [101]:
# ── Configuration ──────────────────────────────────────────────────────────────
BASE_URL      = 'https://crd-app.companiesoffice.govt.nz'
DASHBOARD_URL = f'{BASE_URL}/dashboard/'

# Output folder for CSVs and PDFs — update if running on a different machine
FOLDER     = r'C:\Users\QuyenN\OneDrive - ESNZ\Offline_work\2_GNS\19_LLM_ClimateRisk2026\3-Webscrapping & PDF disclosure'
FOLDER     = r'C:\Users\Quyen\OneDrive - ESNZ\Offline_work\2_GNS\19_LLM_ClimateRisk2026\3-Webscrapping & PDF disclosure'
PDF_FOLDER = os.path.join(FOLDER, 'pdfs_2026','pdfs')
# Chrome path
CHROMEDRIVER_PATH = r"C:\Users\QuyenN\OneDrive - ESNZ\Offline_work\2_GNS\19_LLM_ClimateRisk2026\4-Github2026_LLM\chromedriver-win64\chromedriver.exe"
CHROMEDRIVER_PATH = r"C:\Users\Quyen\OneDrive - ESNZ\Offline_work\2_GNS\19_LLM_ClimateRisk2026\4-Github2026_LLM\chromedriver-win64\chromedriver.exe"

os.makedirs(PDF_FOLDER, exist_ok=True)

# Reporting-period years to target for PDF download
TARGET_YEARS = [2023, 2024, 2025,2026]

print(f"CSV output  : {FOLDER}")
print(f"PDF output  : {PDF_FOLDER}")
print(f"Target years: {TARGET_YEARS}")

CSV output  : C:\Users\Quyen\OneDrive - ESNZ\Offline_work\2_GNS\19_LLM_ClimateRisk2026\3-Webscrapping & PDF disclosure
PDF output  : C:\Users\Quyen\OneDrive - ESNZ\Offline_work\2_GNS\19_LLM_ClimateRisk2026\3-Webscrapping & PDF disclosure\pdfs_2026\pdfs
Target years: [2023, 2024, 2025, 2026]


In [102]:
# Selenium ChromeDriver setup
options = webdriver.ChromeOptions()

# Use new headless mode
options.add_argument("--headless=new")
options.add_argument("--start-maximized")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-blink-features=AutomationControlled")

# PDF download settings
prefs = {
    "download.default_directory": PDF_FOLDER,  # must be absolute path
    "download.prompt_for_download": False,
    "download.directory_upgrade": True,
    "plugins.always_open_pdf_externally": True,
    "safebrowsing.enabled": True,
}
options.add_experimental_option("prefs", prefs)

# Use your local ChromeDriver
driver = webdriver.Chrome(
    service=ChromeService(CHROMEDRIVER_PATH),
    options=options,
)

# Enable downloads in headless mode
driver.execute_cdp_cmd(
    "Page.setDownloadBehavior",
    {
        "behavior": "allow",
        "downloadPath": PDF_FOLDER,
    },
)

print("Chrome driver ready")

Chrome driver ready


The first step is to download the list of all entities 

In [ ]:
# ── Step 1: Scrape the full entity list with dynamic pagination ─────────────────
driver.get(DASHBOARD_URL)
time.sleep(3)

# Detect total pages dynamically instead of hardcoding
try:
    page_btns = WebDriverWait(driver, 10).until(
        EC.presence_of_all_elements_located((By.CSS_SELECTOR, 'a[data-page]'))
    )
    total_pages = max(int(b.get_attribute('data-page')) for b in page_btns)
except Exception:
    total_pages = 19  # safe fallback
print(f"Pages to scrape: {total_pages}")

table_final = pd.DataFrame()

for page_num in range(1, total_pages + 1):
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    try:
        btn = WebDriverWait(driver, 10).until(
            EC.element_to_be_clickable((By.CSS_SELECTOR, f'a[data-page="{page_num}"]'))
        )
        btn.click()

        table = WebDriverWait(driver, 10).until(
            EC.visibility_of_element_located((By.CSS_SELECTOR,
                '#mainContent > div.entitylist > div > div.view-grid.table-responsive.has-pagination > table'))
        )
        table_html = table.get_attribute("outerHTML")
        soup = BeautifulSoup(table_html, 'html.parser')
        rows_html = soup.find_all('tr')

        table_data = []
        for row in rows_html:
            cells = row.find_all(['th', 'td'])
            row_data = [c.get_text(strip=True) for c in cells]
            # Collect entity-page hrefs from every cell in this row
            href_values = []
            for c in cells:
                a = c.find('a')
                if a and str(a.get('href', '')).startswith('/dashboard/reportingentity/?id='):
                    href_values.append(a['href'])
            row_data.append(list(set(href_values)))
            table_data.append(row_data)

        page_df = pd.DataFrame(table_data)
        table_final = pd.concat([table_final, page_df], ignore_index=True)
        print(f"  Page {page_num}/{total_pages} — {len(table_final)} entities so far")

    except Exception as e:
        print(f"  Page {page_num} error: {e}")
        continue

print(f"\nTotal rows scraped: {len(table_final)}")

Pages to scrape: 25
  Page 1/25 — 11 entities so far
  Page 2/25 — 22 entities so far
  Page 3/25 — 33 entities so far
  Page 4/25 — 44 entities so far
  Page 5/25 — 55 entities so far
  Page 6/25 — 66 entities so far
  Page 7/25 — 77 entities so far
  Page 8/25 — 88 entities so far
  Page 9/25 — 99 entities so far
  Page 10/25 — 110 entities so far
  Page 11/25 — 121 entities so far
  Page 12/25 — 132 entities so far
  Page 13/25 — 143 entities so far
  Page 14/25 — 154 entities so far
  Page 15/25 — 165 entities so far
  Page 16/25 — 176 entities so far
  Page 17/25 — 187 entities so far
  Page 18/25 — 193 entities so far
  Page 19/25 — 199 entities so far
  Page 20 error: Message: 
Stacktrace:
	chromedriver!GetHandleVerifier [0x7ff6707636a5+15b35]
	chromedriver!GetHandleVerifier [0x7ff670763710+15ba0]
	chromedriver!(No symbol) [0x7ff6704ad8dd]
	chromedriver!(No symbol) [0x7ff670506c2e]
	chromedriver!(No symbol) [0x7ff670506f3c]
	chromedriver!(No symbol) [0x7ff670558247]
	chromedrive

In [7]:
table_final.columns = ['CRE Name', 'Reporting Entity Status', 'NZBN', 'Actions', 'Link']
table_final.drop(index=0, inplace=True)
table_final.reset_index(drop=True, inplace=True)
table_final.head(5)

,CRE Name,Reporting Entity Status,NZBN,Actions,Link
0,AA INSURANCE LIMITED,Registered,9429040865966,View details,[/dashboard/reportingentity/?id=8b6b5ca5-8394-...
1,AFT PHARMACEUTICALS LIMITED,Registered,9429038010415,View details,[/dashboard/reportingentity/?id=a96b5ca5-8394-...
2,AIA NEW ZEALAND LIMITED,Registered,9429039580948,View details,[/dashboard/reportingentity/?id=8d6b5ca5-8394-...
3,AIG INSURANCE NEW ZEALAND LIMITED,Registered,9429031310048,View details,[/dashboard/reportingentity/?id=4fe2dfb9-81f4-...
4,AIR NEW ZEALAND LIMITED,Registered,9429040402543,View details,[/dashboard/reportingentity/?id=ab6b5ca5-8394-...


In [8]:
today = date.today().strftime("%d/%m/%Y")
table_final.to_csv(os.path.join(FOLDER,'pdfs_2026', 'List_of_reporting_firms_2026.csv'), index=False)
print(f"Saved {len(table_final)} entities on {today}")

Saved 198 entities on 01/05/2026


In [21]:
table_final = pd.read_csv(os.path.join(FOLDER, 'pdfs_2026','List_of_reporting_firms_2026.csv'))

links_final = pd.DataFrame()
for _, row in table_final.iterrows():
    try:
        entity_links = ast.literal_eval(row['Link'])
    except Exception:
        continue
    full_links = [BASE_URL + lnk for lnk in entity_links]
    df = pd.DataFrame({'CRE Name': [row['CRE Name']] * len(full_links), 'Link': full_links})
    links_final = pd.concat([links_final, df], ignore_index=True)

table_final  = table_final.drop(columns=['Link'])
links_final  = links_final.drop_duplicates(subset=['Link']).reset_index(drop=True)
links_final = pd.merge(table_final, links_final, on='CRE Name', how='left')
links_final = links_final[links_final['CRE Name']!= 'CRE Name. sort ascending']
links_final.to_csv(os.path.join(FOLDER,'pdfs_2026', 'List_of_reporting_firms_2026_withlink.csv'), index=False)
print(f"Total entity links: {len(links_final)}")
links_final.head(5)

Total entity links: 180


,CRE Name,Reporting Entity Status,NZBN,Actions,Link
0,AA INSURANCE LIMITED,Registered,9429040865966,View details,https://crd-app.companiesoffice.govt.nz/dashbo...
1,AFT PHARMACEUTICALS LIMITED,Registered,9429038010415,View details,https://crd-app.companiesoffice.govt.nz/dashbo...
2,AIA NEW ZEALAND LIMITED,Registered,9429039580948,View details,https://crd-app.companiesoffice.govt.nz/dashbo...
3,AIG INSURANCE NEW ZEALAND LIMITED,Registered,9429031310048,View details,https://crd-app.companiesoffice.govt.nz/dashbo...
4,AIR NEW ZEALAND LIMITED,Registered,9429040402543,View details,https://crd-app.companiesoffice.govt.nz/dashbo...


### Step 2: Visit each entity page, extract reporting-period text and links

In [35]:
links_final = pd.read_csv(os.path.join(FOLDER,'pdfs_2026', 'List_of_reporting_firms_2026_withlink.csv'))

table_details_final = pd.DataFrame()

for idx, (_, row) in enumerate(links_final.iterrows()):
    firm_name = row['CRE Name']
    if idx % 20 == 0:
        print(f"[{idx+1}/{len(links_final)}] {firm_name}")
    link = row['Link']

    driver.get(link)
    time.sleep(2)

    try:
        table = WebDriverWait(driver, 30).until(
            EC.visibility_of_element_located((By.CSS_SELECTOR, "table"))
        )
        table_html = table.get_attribute("outerHTML")
        soup = BeautifulSoup(table_html, 'html.parser')

        period_data = []
        for tr in soup.find_all('tr')[1:]:
            tds = tr.find_all('td')
            if not tds:
                continue
            a = tds[0].find('a')
            period_data.append({
                'PeriodText': tds[0].get_text(strip=True),
                'PeriodHref': a['href'] if (a and a.get('href')) else None,
            })

        row_data = pd.DataFrame([{
            'Link': link,
            'Company Name': firm_name,
            **item,
        } for item in period_data])



        table_details_final = pd.concat([table_details_final, row_data], ignore_index=True)

    except Exception as e:
        print(f"  Error for {firm_name}: {e}")

table_details_final.to_csv(os.path.join(FOLDER, 'pdfs_2026', 'List_of_reporting_firms_2026_detailed.csv'), index=False)
print(f"\nSaved {len(table_details_final)} entity detail rows")

[1/180] AA INSURANCE LIMITED
[21/180] AUSTRALIAN FOUNDATION INVESTMENT COMPANY LIMITED
[41/180] COMMONWEALTH BANK OF AUSTRALIA
[61/180] FUNDROCK NZ LIMITED
[81/180] KINGFISH LIMITED
[101/180] NAPIER PORT HOLDINGS LIMITED
[121/180] QBE INSURANCE (AUSTRALIA) LIMITED
[141/180] SOUTHERN CROSS MEDICAL CARE SOCIETY
[161/180] TOWER LIMITED

Saved 1261 entity detail rows


### Since the investment schemes have different ways of reporting as compared to firms, we start dividing the ones with schemes only and do a secondary crawling

In [55]:
table_details_final = pd.read_csv(os.path.join(FOLDER, 'pdfs_2026', 'List_of_reporting_firms_2026_detailed.csv'))
print(f"Number of rows : {table_details_final.shape[0]}")

#Remove rows with empty or null PeriodHref
table_details_final = table_details_final[(table_details_final['PeriodHref'].notnull()) & (table_details_final['PeriodHref'] != "") & (table_details_final['PeriodHref'] != "#")]
print(f"Number of rows : {table_details_final.shape[0]}")

#Divide into scheme vs non-scheme entities based on 'Scheme' in PeriodText
table_details_final['Type'] = table_details_final['PeriodText'].apply(
    lambda x: 'companies' if re.search(r'\d{2}-\d{2}-\d{4}', str(x)) else 'investment schemes'
)

#Select only scheme entities
scheme_rows = table_details_final[table_details_final['Type'].str.contains('schemes', case=False)]
print(f"Scheme entities: {scheme_rows.shape[0]}")
scheme_rows.head()

#STart downloading schme sub-entity details
table_details_final_scheme = pd.DataFrame()
for idx, (_, row) in enumerate(scheme_rows.iterrows()):
    firm_name = row['Company Name']
    investment_name = row['PeriodText']
    link = BASE_URL + row['PeriodHref'].strip()
    print(f"[{idx+1}/{len(scheme_rows)}] {firm_name}")

    driver.get(link)
    time.sleep(3)

    try:
        table = WebDriverWait(driver, 30).until(
            EC.visibility_of_element_located((By.CSS_SELECTOR, "table"))
        )
        table_html = table.get_attribute("outerHTML")
        soup = BeautifulSoup(table_html, 'html.parser')


        period_data = []
        for tr in soup.find_all('tr')[1:]:
            tds = tr.find_all('td')
            if not tds:
                continue
            a = tds[0].find('a')
            period_data.append({
                'PeriodText': tds[0].get_text(strip=True),
                'PeriodHref': a['href'] if (a and a.get('href')) else None,
            })

        row_data = pd.DataFrame([{
            'Link': link,
            'Company Name': firm_name,
            'Investment Scheme Name': investment_name,
            **item,
        } for item in period_data])

        page_df = pd.DataFrame(row_data)
        table_details_final_scheme = pd.concat([table_details_final_scheme, page_df], ignore_index=True)

    except Exception as e:
        print(f"  Error: {e}")
table_details_final_scheme.head()
#Drop those that have empty periodhref or periodtext
table_details_final_scheme = table_details_final_scheme[(table_details_final_scheme['PeriodHref'].notnull()) & (table_details_final_scheme['PeriodHref'] != "") & (table_details_final_scheme['PeriodHref'] != "#")]
table_details_final_scheme.to_csv(os.path.join(FOLDER, 'pdfs_2026', 'List_of_investment_scheme_sub-entities_2026.csv'), index=False)
print(f"\nSaved {len(table_details_final_scheme)} scheme sub-entities")


Number of rows : 1261
Number of rows : 516
Scheme entities: 125
[1/125] AMP WEALTH MANAGEMENT NEW ZEALAND LIMITED
[2/125] AMP WEALTH MANAGEMENT NEW ZEALAND LIMITED
[3/125] AMP WEALTH MANAGEMENT NEW ZEALAND LIMITED
[4/125] AMP WEALTH MANAGEMENT NEW ZEALAND LIMITED
[5/125] AMP WEALTH MANAGEMENT NEW ZEALAND LIMITED
[6/125] AMP WEALTH MANAGEMENT NEW ZEALAND LIMITED
[7/125] AMP WEALTH MANAGEMENT NEW ZEALAND LIMITED
[8/125] ANZ INVESTMENT SERVICES (NEW ZEALAND) LIMITED
[9/125] ANZ NEW ZEALAND INVESTMENTS LIMITED
[10/125] ANZ NEW ZEALAND INVESTMENTS LIMITED
[11/125] ANZ NEW ZEALAND INVESTMENTS LIMITED
[12/125] ANZ NEW ZEALAND INVESTMENTS LIMITED
[13/125] ANZ NEW ZEALAND INVESTMENTS LIMITED
[14/125] ASB GROUP INVESTMENTS LIMITED
[15/125] ASB GROUP INVESTMENTS LIMITED
[16/125] BNZ INVESTMENT SERVICES LIMITED
[17/125] BNZ INVESTMENT SERVICES LIMITED
[18/125] BNZ INVESTMENT SERVICES LIMITED
[19/125] BOOSTER INVESTMENT MANAGEMENT LIMITED
[20/125] BOOSTER INVESTMENT MANAGEMENT LIMITED
[21/125] BOOS

### Now combine the list from investment schemes and normal companies and start downloading

In [73]:
#Read the first table 
table_details_final = pd.read_csv(os.path.join(FOLDER, 'pdfs_2026', 'List_of_reporting_firms_2026_detailed.csv'))
print(f"Number of rows : {table_details_final.shape[0]}")

#Remove rows with empty or null PeriodHref
table_details_final = table_details_final[(table_details_final['PeriodHref'].notnull()) & (table_details_final['PeriodHref'] != "") & (table_details_final['PeriodHref'] != "#")]
print(f"Number of rows : {table_details_final.shape[0]}")

#Divide into scheme vs non-scheme entities based on 'Scheme' in PeriodText
table_details_final['Type'] = table_details_final['PeriodText'].apply(
    lambda x: 'companies' if re.search(r'\d{2}-\d{2}-\d{4}', str(x)) else 'investment schemes'
)

links_company_final = table_details_final[table_details_final['Type'].str.contains('companies', case=False)]
print(f"Company entities: {links_company_final.shape[0]}")
links_scheme_final = pd.read_csv(os.path.join(FOLDER, 'pdfs_2026', 'List_of_investment_scheme_sub-entities_2026.csv'))
print(f"Scheme entities: {links_scheme_final.shape[0]}")

#Combine them two
table_details_final_scheme_detail = pd.concat([links_company_final, links_scheme_final], ignore_index=True)
print(f"Total combined entities: {table_details_final_scheme_detail.shape[0]}")

# links_scheme_final['Link'] is a raw string like "/dashboard/...'" — convert to full URL
table_details_final_scheme_detail['DocumentLink'] = table_details_final_scheme_detail['PeriodHref'].apply(lambda x: BASE_URL + x if x else None)

# Fill in the gap for those that have type investment
table_details_final_scheme_detail['Type']= table_details_final_scheme_detail['Type'].fillna('investment schemes')

# Classify submission status
# Note: website now says "Select to view climate statement" (was "Click to view Climate Statement")
SUBMITTED_PHRASE = r'(click|select) to view climate statement'
table_details_final_scheme_detail['status'] = np.where(
    table_details_final_scheme_detail['PeriodText'].str.contains('no records', case=False), 'No records', '')
table_details_final_scheme_detail['status'] = np.where(
    table_details_final_scheme_detail['PeriodText'].str.contains('Due ', case=False),
    'Have records but not submitted', table_details_final_scheme_detail['status'])
table_details_final_scheme_detail['status'] = np.where(
    table_details_final_scheme_detail['PeriodText'].str.contains(SUBMITTED_PHRASE, case=False, regex=True),
    'Have records and have submitted', table_details_final_scheme_detail['status'])

# Get the reporting year
def get_target_year(text, target_years):
    """Return the first year found in text that is in target_years, else None."""
    years = [int(y) for y in re.findall(r'\b\d{2}-\d{2}-(\d{4})\b', str(text))]
    for y in years:
        if y in target_years:
            return y
    return None

table_details_final_scheme_detail['period_year'] = table_details_final_scheme_detail['PeriodText'].apply(
    lambda x: get_target_year(x, TARGET_YEARS))

# Save the combined table
table_details_final_scheme_detail.to_csv(os.path.join(FOLDER, 'pdfs_2026', 'List_of_all_disclosures_as_of_2026.csv'), index=False)
print(f"Saved combined entity details with {table_details_final_scheme_detail.shape[0]} rows")
#table_details_final_scheme_detail.head(5)


Number of rows : 1261
Number of rows : 516
Company entities: 391
Scheme entities: 352
Total combined entities: 743
Saved combined entity details with 743 rows


C:\Users\Quyen\AppData\Local\Temp\ipykernel_14320\418150236.py:38: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  table_details_final_scheme_detail['PeriodText'].str.contains(SUBMITTED_PHRASE, case=False, regex=True),


### Now starting downloading everything 

In [ ]:
from urllib.parse import urlparse

# ── Main download loop ─────────────────────────────────────────────────────────
disclose_rows = pd.read_csv(os.path.join(FOLDER, 'pdfs_2026', 'List_of_all_disclosures_as_of_2026.csv'))

def download_attachments(driver,document_url, company,period_year):
    # Create a subfolder per company
    safe_name = re.sub(r'[^\w\s-]', '', company)[:50].strip().replace(' ', '_') + f"_{period_year}"
    company_folder = os.path.join(PDF_FOLDER, safe_name)
    os.makedirs(company_folder, exist_ok=True)

    # Point Chrome to the company folder
    driver.execute_cdp_cmd("Page.setDownloadBehavior", {
        "behavior": "allow",
        "downloadPath": company_folder,
    })

    driver.get(document_url)
    time.sleep(2)
    soup = BeautifulSoup(driver.page_source, 'html.parser')

    lodgement_links = [
        BASE_URL + a['href']
        for a in soup.find_all('a', href=True)
        if 'lodgement' in a['href'] and 'id=' in a['href']
    ]

    if not lodgement_links:
        return 0

    count = 0
    for lod_url in lodgement_links:
        driver.get(lod_url)
        time.sleep(3)
        for a in driver.find_elements(By.CSS_SELECTOR, 'div.attachment a'):
            a.click()
            time.sleep(2)
            count += 1
    print('Download for {}: {} files'.format(company, count))
    return count



download_log = []
print(f"Downloading PDFs for {len(disclose_rows)} disclosures ({TARGET_YEARS})...\n")

for _, row in disclose_rows[1:].iterrows():
    company    = row['Company Name']
    scheme    = row['Investment Scheme Name']
    document_url = row['DocumentLink']
    year       = row['period_year']

    print(f"{company} ({year})")
    #First download
    count = download_attachments(driver, document_url, company, year)
    #Then log the download
    download_log.append({company: company, 'scheme': scheme, 'year': year, 'document_url': document_url, 'count': count})

log_df = pd.DataFrame(download_log)
log_df.to_csv(os.path.join(FOLDER, 'pdfs_2026', 'download_log_2026.csv'), index=False)
print(f"\n{'='*60}")
print("Download complete.")



AA INSURANCE LIMITED (2024)
Download for AA INSURANCE LIMITED: 2 files
AFT PHARMACEUTICALS LIMITED (2024)
Download for AFT PHARMACEUTICALS LIMITED: 4 files
AFT PHARMACEUTICALS LIMITED (2025)
Download for AFT PHARMACEUTICALS LIMITED: 2 files
AFT PHARMACEUTICALS LIMITED (2026)
AIA NEW ZEALAND LIMITED (2024)
Download for AIA NEW ZEALAND LIMITED: 2 files
AIA NEW ZEALAND LIMITED (2023)
Download for AIA NEW ZEALAND LIMITED: 2 files
AIA NEW ZEALAND LIMITED (2025)
Download for AIA NEW ZEALAND LIMITED: 2 files
AIG INSURANCE NEW ZEALAND LIMITED (2025)
Download for AIG INSURANCE NEW ZEALAND LIMITED: 2 files
AIG INSURANCE NEW ZEALAND LIMITED (2024)
Download for AIG INSURANCE NEW ZEALAND LIMITED: 10 files
AIR NEW ZEALAND LIMITED (2024)
Download for AIR NEW ZEALAND LIMITED: 2 files
AIR NEW ZEALAND LIMITED (2025)
Download for AIR NEW ZEALAND LIMITED: 2 files
ANZ BANK NEW ZEALAND LIMITED (2024)
Download for ANZ BANK NEW ZEALAND LIMITED: 2 files
ANZ BANK NEW ZEALAND LIMITED (2024)
ANZ BANK NEW ZEALAND

In [ ]:
driver.quit()
print("Browser closed.")
print(f"\nPDFs saved to : {PDF_FOLDER}")
print(f"Download log  : {os.path.join(FOLDER, 'download_log_2026.csv')}")